# K-Means Clustering On Wine Profiles

**Purpose:** cluster wines from chemistry measurements and compare the discovered groups with known cultivar labels only after unsupervised model selection.

**Dataset:** `sklearn.datasets.load_wine()` with 13 continuous chemistry features and three known cultivars.

**Method:** standardize features, select `k` by silhouette score, visualize clusters in PCA space, and benchmark the final clusters against labels post hoc.

**Metric:** silhouette score for unsupervised selection; adjusted Rand index only as a final labeled benchmark.

**Headline takeaway:** `k=3` aligns well with the known cultivars, but labels are not used to choose the cluster count.


## Imports And Data Load

The true cultivar labels are loaded for the final benchmark, but the clustering steps treat the feature matrix as unlabeled.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from ml_portfolio.plotting import ACCENT, CAPTION, HIGHLIGHT, MUTED, apply_portfolio_style, save_figure
from sklearn.cluster import KMeans
from sklearn.datasets import load_wine
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "assets").exists() and (PROJECT_ROOT.parent / "assets").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

apply_portfolio_style()

RANDOM_STATE = 42
wine = load_wine()
X = wine.data
y = wine.target
feature_names = wine.feature_names
target_names = wine.target_names

print(f"Rows: {X.shape[0]} | Features: {X.shape[1]} | Known classes: {len(target_names)}")


## Standardize Chemistry Measurements

K-means depends on Euclidean distances, so all chemistry measurements must be put on the same scale before clustering.


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

scaled_preview = pd.DataFrame(X_scaled, columns=feature_names).describe().loc[["mean", "std"]]
display(scaled_preview)


## Sweep Candidate Cluster Counts

Only unsupervised criteria are used here. The adjusted Rand index is intentionally left out of this table so labels cannot influence `k` selection.


In [ ]:
rows = []
for k in range(2, 9):
    model = KMeans(n_clusters=k, n_init=20, random_state=RANDOM_STATE)
    labels = model.fit_predict(X_scaled)
    rows.append(
        {
            "k": k,
            "inertia": model.inertia_,
            "silhouette": silhouette_score(X_scaled, labels),
        }
    )

cluster_scores = pd.DataFrame(rows)
display(cluster_scores)


## Choose k By Silhouette

Silhouette compares cohesion inside clusters with separation between clusters. Higher values are better, while inertia is shown mainly as a supporting elbow-style diagnostic.


In [ ]:
best_k = int(cluster_scores.sort_values("silhouette", ascending=False).iloc[0]["k"])
print(f"Selected k by silhouette score: {best_k}")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(cluster_scores["k"], cluster_scores["inertia"], marker="o")
ax[0].set_title("Inertia by k")
ax[0].set_xlabel("k")
ax[0].set_ylabel("Inertia")
ax[1].plot(cluster_scores["k"], cluster_scores["silhouette"], marker="o", color="tab:green")
ax[1].set_title("Silhouette score by k")
ax[1].set_xlabel("k")
ax[1].set_ylabel("Silhouette")
plt.tight_layout()
plt.show()


## Fit The Selected Clustering Model

The selected model is fit once using the scaled feature matrix.


In [ ]:
kmeans = KMeans(n_clusters=best_k, n_init=20, random_state=RANDOM_STATE)
labels = kmeans.fit_predict(X_scaled)

cluster_counts = pd.Series(labels, name="cluster").value_counts().sort_index().rename("count")
display(cluster_counts.to_frame())


## Profile The Clusters

Cluster centers are converted back to the original feature scale so the groups can be described in domain terms instead of standardized units.


In [ ]:
centers = scaler.inverse_transform(kmeans.cluster_centers_)
center_table = pd.DataFrame(centers, columns=feature_names)
selected_profile_features = ["alcohol", "flavanoids", "color_intensity", "proline"]
display(center_table[selected_profile_features].round(2))


## Visualize Clusters In PCA Space

The PCA plot is a two-dimensional view of a 13-feature clustering result. It is useful for review, but it is not the selection criterion.


In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)
centers_pca = pca.transform(kmeans.cluster_centers_)
explained = pca.explained_variance_ratio_.sum()

print(f"PCA explained variance in 2D view: {explained:.3f}")

fig, ax = plt.subplots(figsize=(6.8, 5.0))
scatter = ax.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=labels,
    cmap="viridis",
    edgecolor="white",
    linewidth=0.4,
    s=45,
)
ax.scatter(centers_pca[:, 0], centers_pca[:, 1], marker="X", s=220, c=HIGHLIGHT, label="Centroids")
ax.set_title(f"Wine clustering at k={best_k} in the 2D PCA view")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
legend = ax.legend(*scatter.legend_elements(), title="Cluster", loc="lower right")
ax.add_artist(legend)
ax.legend(loc="upper right")
ax.text(
    0,
    -0.18,
    f"Wine benchmark; PCA view explains {explained:.1%} of standardized-feature variance.",
    transform=ax.transAxes,
    color=CAPTION,
    fontsize=9,
)
fig.tight_layout()
save_figure(fig, "kmeans_wine_clusters", project_root=PROJECT_ROOT)
plt.show()

## Post-Hoc Label Benchmark

Now that `k` has already been chosen, the known cultivar labels can be used to quantify how closely the unsupervised groups match the benchmark classes.


In [ ]:
ari = adjusted_rand_score(y, labels)
print(f"Adjusted Rand index vs known cultivars: {ari:.3f}")

benchmark = pd.crosstab(
    pd.Series(y, name="known_cultivar").map(dict(enumerate(target_names))),
    pd.Series(labels, name="cluster"),
)
display(benchmark)


## Conclusion

The selected `k=3` solution is consistent with the known three-cultivar structure, but the label benchmark is deliberately post hoc. This keeps the notebook honest about the difference between unsupervised selection and supervised validation.
